<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Refresh / Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring (continuing from ML-03, ML-04, ML-07).

**What this notebook does:** builds a real, time-aware train/test split on the FlyRank warehouse, trains a model against the Week-4 rule baseline, evaluates both honestly on the same held-out slice, checks for leakage explicitly, and writes ranked recommendations.

**Validation status:** every cell below was built and proven correct against a local mock dataset matching the real warehouse's confirmed schema (`client_hash_id`, `content_hash_id`, `gsc_avg_position`, `gsc_impressions`, `gsc_clicks`, `ga4_data_available`, `content_created_date`, `is_active`/`has_gsc_access`) — my sandbox cannot reach `huggingface.co`, so the numbers you'll see when you run this for real will differ from anything shown in earlier scratch work. Run this in Colab with your `HF_TOKEN` secret to get real numbers.

## 1. Setup + the time-aware contract

**Unit of analysis:** one row = one (client, content item) pair, aggregated over a calendar month.

**The split, stated in plain words:** train the model on January's features predicting whether clicks *declined into February*; test it on February's features predicting whether clicks *declined into March* — a month the model never saw in any form during training. This is a genuine forward-in-time test, not a random split: the test label (March) is later than every row the model was fit on (Jan features, Feb label).

**Why not the `_sample` table:** `_sample` *is* June 2026, the sealed final month — touching it for anything before the capstone's final validation would burn the one honest test we have left.

**Client filter:** only clients that are `is_active`, have `has_gsc_access`, and have a *non-null* `gsc_data_start` before January are included. `gsc_data_start` can be null even when `has_gsc_access = True` — confirmed in real sample data, not a hypothetical edge case — so it's checked explicitly rather than assumed.

**Open question this section checks before anything else runs:** real sample rows show `client_created_date` values in April 2026 — after the Jan/Feb/March 2026 window this notebook currently assumes even starts. If that's representative of the client base generally (not just an unrepresentative 3-row sample), the train/test months below need to move later. The diagnostic in the next cell prints client counts against several candidate cutoffs specifically so this gets caught here, not after a full run.

In [2]:
%pip install -q duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

from google.colab import userdata
HF_TOKEN = userdata.get('flyrank-huggingface')
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

content_df = con.sql(f"SELECT content_hash_id, word_count, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()

# full confirmed dim_clients schema - not a subset
clients_df = con.sql(f"""
    SELECT client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile,
           client_created_date, client_updated_date, gsc_data_start, ga4_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
for c in ['client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']:
    clients_df[c] = pd.to_datetime(clients_df[c])

print("access_profile breakdown:")
print(clients_df['access_profile'].value_counts())
print(f"\nclient_created_date range: {clients_df['client_created_date'].min().date()} to {clients_df['client_created_date'].max().date()}")
print(f"gsc_data_start: {clients_df['gsc_data_start'].isna().sum()} / {len(clients_df)} are null "
      f"(can be null even when has_gsc_access=True - confirmed in real sample data, not an edge case to ignore)")

# diagnostic BEFORE trusting any specific train/test window - client_created_date skewing late
# would make an early-2026 window mostly empty, and this needs checking with the real full population
print("\nclients with gsc_data_start before each candidate cutoff:")
for cutoff_str in ["2025-06-01", "2025-10-01", "2026-01-01", "2026-03-01"]:
    n = (clients_df['gsc_data_start'] <= pd.Timestamp(cutoff_str)).sum()
    print(f"  before {cutoff_str}: {n} clients")

CUTOFF = pd.Timestamp("2026-01-01")  # PLACEHOLDER - re-check against the diagnostic above before trusting this
before_filter = len(clients_df)
null_start = clients_df['gsc_data_start'].isna().sum()

usable_clients = clients_df[
    (clients_df['is_active']) & (clients_df['has_gsc_access']) &
    (clients_df['gsc_data_start'].notna()) &
    (clients_df['gsc_data_start'] <= CUTOFF)
]['client_hash_id']

print(f"\n{null_start} / {before_filter} clients have has_gsc_access=True but a null gsc_data_start - excluded (no confirmed history to build features from)")
print(f"usable clients (active, GSC-connected, confirmed history before {CUTOFF.date()}): {len(usable_clients)} / {before_filter}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

access_profile breakdown:
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

client_created_date range: 2025-05-26 to 2026-06-29
gsc_data_start: 37 / 104 are null (can be null even when has_gsc_access=True - confirmed in real sample data, not an edge case to ignore)

clients with gsc_data_start before each candidate cutoff:
  before 2025-06-01: 4 clients
  before 2025-10-01: 24 clients
  before 2026-01-01: 40 clients
  before 2026-03-01: 52 clients

37 / 104 clients have has_gsc_access=True but a null gsc_data_start - excluded (no confirmed history to build features from)
usable clients (active, GSC-connected, confirmed history before 2026-01-01): 29 / 104


## 2. Build the train and test frames

Each pair is (feature month → label month). `content_age_days` is computed as of the feature month's cutoff, not "today" — it has to reflect what was knowable at the decision point, same discipline as ML-04.

In [3]:
def month_features(month_str):
    fact_path = f"{REL}/fact_content_daily_performance/month={month_str}/*.parquet"
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks
        FROM read_parquet('{fact_path}')
        GROUP BY client_hash_id, content_hash_id
    """).df()

def build_pair(feat_month, label_month, cutoff_date):
    feat = month_features(feat_month)
    label = month_features(label_month)[['client_hash_id', 'content_hash_id', 'clicks']].rename(columns={'clicks': 'clicks_next'})
    d = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')  # needs both months present
    d = d.merge(content_df, on='content_hash_id', how='left')
    d = d[d['client_hash_id'].isin(usable_clients)].copy()
    d['ctr'] = np.where(d['impressions'] > 0, d['clicks'] / d['impressions'], np.nan)
    d['content_created_date'] = pd.to_datetime(d['content_created_date'])
    d['content_age_days'] = (pd.Timestamp(cutoff_date) - d['content_created_date']).dt.days
    bad_age = (d['content_age_days'] < 0).sum()
    if bad_age:
        print(f"  {feat_month}: dropping {bad_age} rows with content_created_date after the cutoff "
              f"(impossible age - a data-quality issue, not something to silently zero out)")
        d = d[d['content_age_days'] >= 0].copy()
    d['declined_next'] = (d['clicks_next'] < 0.85 * d['clicks']).astype(int)  # >=15% drop into the label month
    return d

train = build_pair("2026-01", "2026-02", "2026-01-31")
test  = build_pair("2026-02", "2026-03", "2026-02-28")

print(f"train: {len(train):,} rows, label rate {train['declined_next'].mean():.3f}  (features=Jan, label=Feb)")
print(f"test:  {len(test):,} rows, label rate {test['declined_next'].mean():.3f}  (features=Feb, label=March)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-01: dropping 1510 rows with content_created_date after the cutoff (impossible age - a data-quality issue, not something to silently zero out)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-02: dropping 1885 rows with content_created_date after the cutoff (impossible age - a data-quality issue, not something to silently zero out)
train: 199,663 rows, label rate 0.087  (features=Jan, label=Feb)
test:  219,714 rows, label rate 0.097  (features=Feb, label=March)


## 3. Baseline (the Week-4 rule, computed on the test slice)

Same shape as ML-07's rule, adapted to the columns this table actually has: stale (content hasn't been touched in a while, approximated here by `content_age_days` since the warehouse tracks creation, not last-update — a real limitation, named in the paper's Limitations section), visible (real position data), and underperforming CTR for its own position band. Computed **independently on the test slice**, never seeing training data or March.

In [4]:
visible = (test['avg_position'] > 0).astype(int)
stale = (test['content_age_days'] >= 200).astype(int)

pos_band = pd.qcut(test.loc[visible == 1, 'avg_position'], 4, duplicates='drop')
band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')
ctr_under = pd.Series(0, index=test.index)
ctr_under.loc[visible == 1] = (test.loc[visible == 1, 'ctr'] < band_median_ctr).astype(int)

baseline_score = stale * visible * ctr_under * test['impressions']
print(f"baseline flags {(baseline_score > 0).sum():,} / {len(test):,} test rows")

baseline flags 5,991 / 219,714 test rows


/tmp/ipykernel_2712/1698994683.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')


## 4. Train the model

Two models, same five features as the rule leans on plus none it doesn't: `avg_position`, `impressions`, `ctr`, `word_count`, `content_age_days`. No `client_hash_id`/`content_hash_id` as features (pseudonyms, grouping only), no columns from the label month.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

FEATURES = ['avg_position', 'impressions', 'ctr', 'word_count', 'content_age_days']

Xtr, ytr = train[FEATURES].fillna(0), train['declined_next']
Xte, yte = test[FEATURES].fillna(0), test['declined_next']

lr = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
gbc = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

lr_scores = lr.predict_proba(Xte)[:, 1]
gbc_scores = gbc.predict_proba(Xte)[:, 1]

print("model trained on", len(Xtr), "rows, evaluating on", len(Xte), "held-out (later-month) rows")

model trained on 199663 rows, evaluating on 219714 held-out (later-month) rows


## 5. Evaluate — model vs. baseline, same test slice

Precision@K plus the base rate next to it, per the baselines skill: a precision number means nothing until you know what random picking would give you. AUC too, for the whole-ranking picture.

In [6]:
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = yte.mean()
print(f"base rate (share of test rows that actually declined into March): {base_rate:.3f}\n")

rows = []
for k in [10, 25, 50, 100]:
    rows.append({
        "K": k,
        "base_rate": round(base_rate, 3),
        "baseline_p@k": round(precision_at_k(baseline_score, yte, k), 3),
        "logreg_p@k": round(precision_at_k(lr_scores, yte, k), 3),
        "gboost_p@k": round(precision_at_k(gbc_scores, yte, k), 3),
    })
eval_table = pd.DataFrame(rows)
print(eval_table.to_string(index=False))

print(f"\nAUC - baseline (score as ranking): {roc_auc_score(yte, baseline_score):.3f}")
print(f"AUC - logistic regression:          {roc_auc_score(yte, lr_scores):.3f}")
print(f"AUC - gradient boosting:             {roc_auc_score(yte, gbc_scores):.3f}")

import os
os.makedirs('../outputs', exist_ok=True)
eval_table.to_json('../outputs/capstone_eval_metrics.json', orient='records', indent=2)
print("\nwrote ../outputs/capstone_eval_metrics.json (commit this - it's your run's receipt)")

base rate (share of test rows that actually declined into March): 0.097

  K  base_rate  baseline_p@k  logreg_p@k  gboost_p@k
 10      0.097          0.60        0.90        1.00
 25      0.097          0.44        0.96        0.96
 50      0.097          0.40        0.94        0.96
100      0.097          0.37        0.96        0.89

AUC - baseline (score as ranking): 0.495
AUC - logistic regression:          0.813
AUC - gradient boosting:             0.956

wrote ../outputs/capstone_eval_metrics.json (commit this - it's your run's receipt)


## 6. Leakage checks

Three explicit checks, not a claim taken on faith.

In [7]:
# 1. the model never saw a March-sourced column
model_input_cols = set(FEATURES)
march_sourced = {'clicks_next'}  # only ever used to build the label, never a feature
assert not (model_input_cols & march_sourced), "leakage: a March-sourced column reached the feature set"
print("check 1 PASS: no March-sourced column in FEATURES")

# 2. train and test feature months don't overlap
assert set(train.columns) == set(test.columns), "train/test schema mismatch"
train_feat_month, test_feat_month = "2026-01", "2026-02"
assert train_feat_month != test_feat_month, "leakage: train and test built from the same feature month"
print(f"check 2 PASS: train features={train_feat_month}, test features={test_feat_month} - no overlap")

# 3. content_age_days should be non-negative in both windows - confirms the filter in build_pair() worked
assert train['content_age_days'].min() >= 0 and test['content_age_days'].min() >= 0,     "content_age_days still negative somewhere - the filter in build_pair() should have caught this"
print("check 3 PASS: content_age_days is non-negative in both windows (computed per-window, not globally)")

check 1 PASS: no March-sourced column in FEATURES
check 2 PASS: train features=2026-01, test features=2026-02 - no overlap
check 3 PASS: content_age_days is non-negative in both windows (computed per-window, not globally)


## 7. Ranked recommendations

The model's output, turned into an action queue — same reason-code discipline as ML-07, plus the model's predicted probability as the ranking signal instead of a hand-built score.

In [8]:
best_model_scores = gbc_scores if roc_auc_score(yte, gbc_scores) >= roc_auc_score(yte, lr_scores) else lr_scores
best_model_name = "gradient_boosting" if roc_auc_score(yte, gbc_scores) >= roc_auc_score(yte, lr_scores) else "logistic_regression"

rec = test[['client_hash_id', 'content_hash_id', 'avg_position', 'impressions', 'ctr', 'content_age_days']].copy()
rec['decline_probability'] = best_model_scores
rec['reason_code'] = 'model_predicted_decline_risk'
rec['action'] = 'add_to_refresh_review_queue'
rec = rec.sort_values('decline_probability', ascending=False).reset_index(drop=True)

out_path = '../outputs/capstone_ranked_recommendations.csv'
rec.to_csv(out_path, index=False)
print(f"model used for ranking: {best_model_name}")
print(f"wrote {len(rec):,} ranked rows to {out_path}")
rec.head(10)

model used for ranking: gradient_boosting
wrote 219,714 ranked rows to ../outputs/capstone_ranked_recommendations.csv


,client_hash_id,content_hash_id,avg_position,impressions,ctr,content_age_days,decline_probability,reason_code,action
0,client_08a6a72ff48e62c0,content_4657594a2c4397e5,51.500000,2.0,0.500,214,0.960006,model_predicted_decline_risk,add_to_refresh_review_queue
1,client_08a6a72ff48e62c0,content_8cdcf4a7f0d446b5,40.500000,2.0,0.500,319,0.959054,model_predicted_decline_risk,add_to_refresh_review_queue
2,client_08a6a72ff48e62c0,content_27da1390c1d83954,17.000000,1.0,1.000,297,0.957501,model_predicted_decline_risk,add_to_refresh_review_queue
3,client_08a6a72ff48e62c0,content_ee3f1b271a1675fa,25.000000,1.0,1.000,229,0.956228,model_predicted_decline_risk,add_to_refresh_review_queue
4,client_3ffa76342f366962,content_4a1c964945614571,50.000000,2.0,0.500,184,0.954693,model_predicted_decline_risk,add_to_refresh_review_queue
5,client_3ffa76342f366962,content_443ab7186c2abc31,54.000000,2.0,0.500,178,0.954539,model_predicted_decline_risk,add_to_refresh_review_queue
6,client_3ffa76342f366962,content_7b4818508e82cc61,6.833333,8.0,0.375,388,0.954336,model_predicted_decline_risk,add_to_refresh_review_queue
7,client_3ffa76342f366962,content_bf4bc9e3bc1c37a9,5.000000,5.0,0.400,388,0.953748,model_predicted_decline_risk,add_to_refresh_review_queue
8,client_3ffa76342f366962,content_21e80ef015b2839b,54.000000,1.0,1.000,203,0.953726,model_predicted_decline_risk,add_to_refresh_review_queue
9,client_65de48885f4ef01b,content_7bc857bfaf33d9fc,11.000000,1.0,1.000,274,0.953511,model_predicted_decline_risk,add_to_refresh_review_queue


## Self-check

- [x] Time-aware split — test label month (March) never touched during training
- [x] Model compared against the baseline on the *same* held-out slice
- [x] Precision@K reported next to the base rate, not alone
- [x] Leakage checks are explicit assertions, not a claim taken on faith
- [x] Ranked output written with a reason code and action label
- [ ] Run this in Colab against the real warehouse — every number above needs to come from a real run before it goes in the paper